# Sentiment directions fitted at adjective and final-token positions

This Colab notebook runs the complete sentiment-position comparison for **GPT-2 Small** and **Qwen3-0.6B Base**. It fits mean difference, logistic regression, and one-dimensional DAS directions at the adjective and final prompt-token positions for every non-embedding residual boundary. It then performs all-token directional patching on ToyMovieReview adjectives, verbs, adverbs, and the model-specific SST directed pairs.

All direction artifacts, result tables, provenance, and figures are written directly to one timestamped Google Drive directory. The notebook calls the `sentiment_geometry` package; it does not reimplement activation extraction, fitting, patching, metrics, or artifact validation.

## Before running

1. Choose **Runtime → Change runtime type → T4 GPU** or a larger GPU.
2. Have a Hugging Face token with read access to the SST dataset repository ready. The notebook will request it through a hidden input prompt.
3. Leave `RESUME_RUN_ID = None` for a new timestamped run. After a disconnection, paste the previous directory name into `RESUME_RUN_ID` to reuse compatible direction checkpoints.
4. The full sweep fits 240 directions and performs causal evaluation for each one. It can take a long time, particularly for DAS.

## 1. User settings

In [1]:
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional commit or tag. Existing checkouts must already match it.

DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
TIMEZONE_NAME = "America/Chicago"
RESUME_RUN_ID = None  # Example: "2026-09-05_18-42_CDT"

DEVICE = "cuda"
DTYPE = "auto"
BATCH_SIZE = 16
RUN_EXPERIMENT = True

## 2. Clone and install the project

An existing checkout is reused. If `PROJECT_REVISION` is set, the notebook verifies the checkout rather than silently changing it. After the editable installation, the repository root is added explicitly to `sys.path`; this project has no `src/` directory. The cell then verifies the package origin and imports every `sentiment_geometry` module as an early installation check.

In [2]:
import importlib
import os
import pkgutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

# Show installation output so dependency or editable-install failures are visible.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)

# The package now lives at PROJECT_ROOT/sentiment_geometry, not PROJECT_ROOT/src.
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
importlib.invalidate_caches()

sentiment_geometry = importlib.import_module("sentiment_geometry")
expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(sentiment_geometry.__file__).resolve().parent
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )

module_names = sorted(
    module.name
    for module in pkgutil.walk_packages(
        sentiment_geometry.__path__, prefix="sentiment_geometry."
    )
    if module.name != "sentiment_geometry.__main__"
)
for module_name in module_names:
    importlib.import_module(module_name)

required_apis = {
    "sentiment_geometry.experiments": {
        "SentimentPositionExperimentConfig",
        "run_sentiment_position_comparison",
        "audit_direction_artifacts",
    },
    "sentiment_geometry.persistence": {
        "RunArtifactStore",
        "maybe_mount_google_drive",
        "prepare_timestamped_run",
    },
}
for module_name, api_names in required_apis.items():
    module = importlib.import_module(module_name)
    missing_apis = sorted(name for name in api_names if not hasattr(module, name))
    if missing_apis:
        raise ImportError(
            f"{module_name} is missing {missing_apis}. "
            "The checkout is older than this notebook; use a fresh runtime."
        )

print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)
print(f"Successfully imported {len(module_names) + 1} package modules.")

Project commit: c7e16f0ea888454724b8131be0b0a0c56194f744
Imported package from: /content/sentiment-manifold/sentiment_geometry
Successfully imported 58 package modules.


## 3. Mount Google Drive and prepare the run

A new run uses a readable minute-level name such as `2026-09-05_18-42_CDT`. The same minute cannot be reused accidentally. Resume requires the exact prior run ID.

In [3]:
import torch

from sentiment_geometry.persistence import maybe_mount_google_drive, prepare_timestamped_run

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")

maybe_mount_google_drive(True)
if "RUN_LAYOUT" not in globals() or RESUME_RUN_ID is not None:
    RUN_LAYOUT = prepare_timestamped_run(
        DRIVE_STORAGE_ROOT,
        experiment_name="sentiment-position-comparison",
        timezone_name=TIMEZONE_NAME,
        resume_run_id=RESUME_RUN_ID,
    )

print("Run ID:             ", RUN_LAYOUT.run_id)
print("Run directory:      ", RUN_LAYOUT.root)
print("Result tables:      ", RUN_LAYOUT.results_dir)
print("Direction artifacts:", RUN_LAYOUT.directions_dir)
print("Figures:            ", RUN_LAYOUT.figures_dir)
print("GPU:                ", torch.cuda.get_device_name(0))

Mounted at /content/drive
Run ID:              2026-09-05_23-18_CDT
Run directory:       /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT
Result tables:       /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/results
Direction artifacts: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/directions
Figures:             /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures
GPU:                 NVIDIA L4


## 4. Read the Hugging Face secret

As in the preprocessing notebook, `HF_TOKEN` is requested with `getpass`, cached only in a notebook-owned runtime dictionary, and removed after the experiment. Its value is never printed or written to Drive.

In [4]:
from getpass import getpass

from huggingface_hub import HfApi

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): ").strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def delete_runtime_secret(name):
    value = _RUNTIME_SECRETS.pop(name, None)
    if value is not None:
        del value

_token = get_runtime_secret("HF_TOKEN")
hf_account = HfApi(token=_token).whoami()["name"]
del _token
print(f"Authenticated to Hugging Face as {hf_account}. Token value was not displayed.")

Authenticated to Hugging Face as kokolamba. Token value was not displayed.


## 5. Load and lock the complete experiment configuration

This cell explicitly fixes the two models, three fitting methods, two fitting positions, every boundary from `1..n_layers`, all-token patching, and the mandatory Toy adjective/verb/adverb plus SST evaluation panels. Boundary 0 is excluded because it is the embedding residual.

In [5]:
from dataclasses import asdict

import pandas as pd
import yaml
from IPython.display import display

from sentiment_geometry.experiments import SentimentPositionExperimentConfig
from sentiment_geometry.experiments.sentiment_position.config import REQUIRED_TOY_EVALUATIONS

CONFIG_PATH = PROJECT_ROOT / "configs/sentiment_position_comparison.yaml"
config = SentimentPositionExperimentConfig.load(CONFIG_PATH)

expected_models = ["gpt2-small", "qwen-0.6b"]
if [model.name for model in config.models] != expected_models:
    raise RuntimeError(f"Expected exactly {expected_models}; got {[m.name for m in config.models]}")

for model in config.models:
    model.device = DEVICE
    model.dtype = DTYPE
    model.batch_size = BATCH_SIZE

config.sweep.layers = "all_non_embedding"
config.sweep.methods = ["mean_diff", "logistic_regression", "das"]
config.sweep.fit_positions = ["adjective", "final"]
config.sweep.output_dir = str(RUN_LAYOUT.results_dir)
config.sweep.checkpoint_dir = str(RUN_LAYOUT.directions_dir)
config.sweep.resume = True
config.data.sst_max_directed_cases = None
config.validate()

if tuple(REQUIRED_TOY_EVALUATIONS) != ("toy_adjectives", "toy_verbs", "toy_adverbs"):
    raise RuntimeError("The mandatory Toy evaluation contract has changed.")

requested_config_path = RUN_LAYOUT.root / "requested_config.yaml"
requested_config_path.write_text(
    yaml.safe_dump(config.to_dict(), sort_keys=False), encoding="utf-8"
)
RUN_LAYOUT.update_manifest(
    status="configured",
    metadata={
        "models": [asdict(model) for model in config.models],
        "methods": config.sweep.methods,
        "fit_positions": config.sweep.fit_positions,
        "layers": config.sweep.layers,
        "required_evaluations": [*REQUIRED_TOY_EVALUATIONS, "sst"],
        "configuration": requested_config_path.name,
    },
)

display(
    pd.DataFrame(
        [
            {
                "model": model.name,
                "hub_model": model.hub_name,
                "revision": model.revision,
                "device": model.device,
                "dtype": model.dtype,
                "batch_size": model.batch_size,
            }
            for model in config.models
        ]
    )
)
print("Methods:", config.sweep.methods)
print("Fitting positions:", config.sweep.fit_positions)
print("Layers: 1..n_layers (embedding boundary 0 excluded)")
print("Evaluations:", [*REQUIRED_TOY_EVALUATIONS, "sst"])

,model,hub_model,revision,device,dtype,batch_size
0,gpt2-small,gpt2,607a30d783dfa663caf39e06633721c8d4cfcd7e,cuda,auto,16
1,qwen-0.6b,Qwen/Qwen3-0.6B-Base,da87bfb608c14b7cf20ba1ce41287e8de496c0cd,cuda,auto,16


Methods: ['mean_diff', 'logistic_regression', 'das']
Fitting positions: ['adjective', 'final']
Layers: 1..n_layers (embedding boundary 0 excluded)
Evaluations: ['toy_adjectives', 'toy_verbs', 'toy_adverbs', 'sst']


## 6. Record the software environment

In [6]:
import platform
from importlib.metadata import version

from sentiment_geometry.persistence import RunArtifactStore

environment = {
    "project_commit": project_commit,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": version("torch"),
    "transformers": version("transformers"),
    "datasets": version("datasets"),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
RunArtifactStore(RUN_LAYOUT.root).write_json("environment.json", environment)
display(pd.Series(environment, name="value").to_frame())

,value
project_commit,c7e16f0ea888454724b8131be0b0a0c56194f744
python,3.12.13
platform,Linux-6.6.122+-x86_64-with-glibc2.35
torch,2.11.0+cu128
transformers,4.57.6
datasets,4.0.0
numpy,2.0.2
pandas,2.2.2
device,cuda
gpu,NVIDIA L4


## 7. Run both models

The package processes GPT-2 Small first, releases its model memory, and then processes Qwen3-0.6B Base. Every fitted direction is saved under `directions/` immediately. The CSV tables under `results/` are refreshed after every layer.

In [7]:
from sentiment_geometry.experiments import run_sentiment_position_comparison

if not RUN_EXPERIMENT:
    print("Experiment execution is disabled. Set RUN_EXPERIMENT = True in the settings cell.")
else:
    previous_hf_token = os.environ.get(config.data.hf_token_env)
    os.environ[config.data.hf_token_env] = get_runtime_secret("HF_TOKEN")
    RUN_LAYOUT.update_manifest(status="running")
    try:
        completed_results_dir = run_sentiment_position_comparison(config)
        if completed_results_dir.resolve() != RUN_LAYOUT.results_dir.resolve():
            raise RuntimeError(f"Unexpected results directory: {completed_results_dir}")
        RUN_LAYOUT.update_manifest(status="experiment-completed")
    except BaseException as error:
        RUN_LAYOUT.update_manifest(
            status="failed", metadata={"failure_type": type(error).__name__}
        )
        raise
    finally:
        if previous_hf_token is None:
            os.environ.pop(config.data.hf_token_env, None)
        else:
            os.environ[config.data.hf_token_env] = previous_hf_token
        delete_runtime_secret("HF_TOKEN")

    print("Experiment results saved to:", completed_results_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

tigges_qwen_0_6b_directed_pairs/test-000(…):   0%|          | 0.00/112k [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

qwen-0.6b sentiment-position boundaries:   0%|          | 0/28 [00:00<?, ?it/s]

Experiment results saved to: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/results


## 8. Audit all saved sentiment directions

The audit reconstructs the expected method × position × layer grid from each model's resolved configuration, checks the metadata for omissions or duplicates, and verifies that every referenced `.npz` file exists on Drive. The pinned architectures should produce 72 GPT-2 directions and 168 Qwen directions.

In [8]:
from datetime import datetime
from zoneinfo import ZoneInfo

from sentiment_geometry.experiments import audit_direction_artifacts

try:
    direction_audit = audit_direction_artifacts(RUN_LAYOUT.results_dir)
    display(direction_audit.summary)
    direction_audit.require_complete()
    if direction_audit.expected_total != 240:
        raise RuntimeError(
            f"Expected 240 direction artifacts for the pinned architectures; "
            f"the resolved run expected {direction_audit.expected_total}."
        )
except BaseException as error:
    RUN_LAYOUT.update_manifest(
        status="audit-failed", metadata={"audit_failure_type": type(error).__name__}
    )
    raise
else:
    completed_at = datetime.now(ZoneInfo(RUN_LAYOUT.timezone_name)).isoformat(timespec="minutes")
    RUN_LAYOUT.update_manifest(
        status="completed",
        metadata={
            "completed_at": completed_at,
            "direction_artifacts": direction_audit.expected_total,
            "direction_audit_complete": True,
        },
    )
    print(f"Verified all {direction_audit.expected_total} direction artifacts on Drive.")

,model,expected_artifacts,recorded_artifacts,existing_artifacts,missing_combinations,unexpected_combinations,duplicate_records,missing_files,complete
0,gpt2-small,72,72,72,0,0,0,0,True
1,qwen-0.6b,168,168,168,0,0,0,0,True


Verified all 240 direction artifacts on Drive.


## 9. Load the saved CSV tables

All analysis below begins by reopening the saved tables from Drive.

In [9]:
MODEL_NAMES = ["gpt2-small", "qwen-0.6b"]

def load_saved_table(filename):
    tables = []
    for model_name in MODEL_NAMES:
        path = RUN_LAYOUT.results_dir / model_name / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        table = pd.read_csv(path)
        if "model" not in table.columns:
            table.insert(0, "model", model_name)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)

dataset_summary = load_saved_table("dataset_summary.csv")
metrics = load_saved_table("metrics.csv")
best_layers = load_saved_table("best_layers.csv")
direction_similarities = load_saved_table("direction_similarities.csv")
direction_metadata = load_saved_table("direction_metadata.csv")

display(dataset_summary)
print(f"Loaded {len(metrics):,} aggregate metric rows from Drive.")
print(f"Loaded {len(direction_metadata):,} direction metadata rows from Drive.")

,model,dataset,role,n_examples,n_directed_cases
0,gpt2-small,toy_train,direction_fitting,55,48
1,gpt2-small,toy_adjectives,causal_evaluation,30,28
2,gpt2-small,toy_verbs,causal_evaluation,8,6
3,gpt2-small,toy_adverbs,causal_evaluation,39,34
4,gpt2-small,sst,causal_evaluation,304,304
5,qwen-0.6b,toy_train,direction_fitting,54,46
6,qwen-0.6b,toy_adjectives,causal_evaluation,30,28
7,qwen-0.6b,toy_verbs,causal_evaluation,7,4
8,qwen-0.6b,toy_adverbs,causal_evaluation,34,30
9,qwen-0.6b,sst,causal_evaluation,302,302


Loaded 960 aggregate metric rows from Drive.
Loaded 240 direction metadata rows from Drive.


## 10. Best logit-difference and logit-flip percentages across layers

These are descriptive maxima for each model × method × fitting position × evaluation dataset × metric. They are not treated as validation-selected layers for a later confirmatory claim.

In [10]:
best_display = best_layers[
    ["model", "dataset", "fit_position", "method", "metric", "layer", "value_percent"]
].sort_values(["model", "dataset", "fit_position", "method", "metric"])
display(best_display.reset_index(drop=True))

,model,dataset,fit_position,method,metric,layer,value_percent
0,gpt2-small,sst,adjective,das,logit_difference,6,49.657534
1,gpt2-small,sst,adjective,das,logit_flip,6,61.206901
2,gpt2-small,sst,adjective,logistic_regression,logit_difference,6,45.890411
3,gpt2-small,sst,adjective,logistic_regression,logit_flip,6,50.000000
4,gpt2-small,sst,adjective,mean_diff,logit_difference,6,40.753425
...,...,...,...,...,...,...,...
91,qwen-0.6b,toy_verbs,final,das,logit_flip,17,100.000000
92,qwen-0.6b,toy_verbs,final,logistic_regression,logit_difference,26,112.770566
93,qwen-0.6b,toy_verbs,final,logistic_regression,logit_flip,18,100.000000
94,qwen-0.6b,toy_verbs,final,mean_diff,logit_difference,26,111.038962


## 11. Direction similarity at first, middle, and last boundaries

The first table compares fitting methods within the same activation position. The second compares adjective versus final-token directions for the same method. Absolute cosine is shown as the primary similarity; signed cosine remains available for orientation auditing.

In [11]:
within_position = direction_similarities[
    (direction_similarities["position_a"] == direction_similarities["position_b"])
    & (direction_similarities["method_a"] < direction_similarities["method_b"])
][
    ["model", "layer", "position_a", "method_a", "method_b", "absolute_cosine", "signed_cosine"]
].rename(columns={"position_a": "fit_position"})

between_positions = direction_similarities[
    (direction_similarities["position_a"] == "adjective")
    & (direction_similarities["position_b"] == "final")
    & (direction_similarities["method_a"] == direction_similarities["method_b"])
][
    ["model", "layer", "method_a", "absolute_cosine", "signed_cosine"]
].rename(columns={"method_a": "method"})

print("Method-to-method similarity within each fitting position")
display(within_position.sort_values(["model", "layer", "fit_position", "method_a", "method_b"]))
print("Adjective-to-final similarity for each fitting method")
display(between_positions.sort_values(["model", "layer", "method"]))

Method-to-method similarity within each fitting position


,model,layer,fit_position,method_a,method_b,absolute_cosine,signed_cosine
13,gpt2-small,1,adjective,das,logistic_regression,0.857975,0.857975
12,gpt2-small,1,adjective,das,mean_diff,0.889583,0.889583
6,gpt2-small,1,adjective,logistic_regression,mean_diff,0.959771,0.959771
34,gpt2-small,1,final,das,logistic_regression,0.667550,0.667550
33,gpt2-small,1,final,das,mean_diff,0.677016,0.677016
27,gpt2-small,1,final,logistic_regression,mean_diff,0.982346,0.982346
49,gpt2-small,6,adjective,das,logistic_regression,0.781860,0.781860
48,gpt2-small,6,adjective,das,mean_diff,0.820672,0.820672
42,gpt2-small,6,adjective,logistic_regression,mean_diff,0.942894,0.942894
70,gpt2-small,6,final,das,logistic_regression,0.744867,0.744867


Adjective-to-final similarity for each fitting method


,model,layer,method,absolute_cosine,signed_cosine
17,gpt2-small,1,das,0.044743,0.044743
10,gpt2-small,1,logistic_regression,0.102176,0.102176
3,gpt2-small,1,mean_diff,0.118289,0.118289
53,gpt2-small,6,das,0.173992,0.173992
46,gpt2-small,6,logistic_regression,0.016645,0.016645
39,gpt2-small,6,mean_diff,0.026646,0.026646
89,gpt2-small,12,das,0.015004,-0.015004
82,gpt2-small,12,logistic_regression,0.344459,0.344459
75,gpt2-small,12,mean_diff,0.336277,0.336277
125,qwen-0.6b,1,das,0.068905,0.068905


## 12. Replot causal metrics from the saved CSVs

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
metric_specs = {
    "logit_difference_percent": "Logit difference (%)",
    "logit_flip_percent": "Logit flip (%)",
}
for model_name in MODEL_NAMES:
    model_metrics = metrics[metrics["model"] == model_name]
    model_figure_dir = RUN_LAYOUT.figures_dir / model_name
    model_figure_dir.mkdir(parents=True, exist_ok=True)
    for metric_column, y_label in metric_specs.items():
        grid = sns.relplot(
            data=model_metrics,
            x="layer",
            y=metric_column,
            hue="method",
            style="fit_position",
            col="dataset",
            col_wrap=2,
            kind="line",
            markers=True,
            facet_kws={"sharey": False},
            height=3.5,
            aspect=1.35,
        )
        grid.set_axis_labels("Residual boundary", y_label)
        grid.fig.suptitle(f"{model_name}: {y_label} across layers", y=1.03)
        figure_path = model_figure_dir / f"{metric_column}_by_layer.png"
        grid.savefig(figure_path, dpi=180, bbox_inches="tight")
        plt.show()
        plt.close(grid.fig)
        print("Saved:", figure_path)

Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/gpt2-small/logit_difference_percent_by_layer.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/gpt2-small/logit_flip_percent_by_layer.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/qwen-0.6b/logit_difference_percent_by_layer.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/qwen-0.6b/logit_flip_percent_by_layer.png


## 13. Replot the first/middle/last cosine matrices

In [13]:
for model_name in MODEL_NAMES:
    model_similarity = direction_similarities[direction_similarities["model"] == model_name]
    model_figure_dir = RUN_LAYOUT.figures_dir / model_name
    for layer in sorted(model_similarity["layer"].unique()):
        layer_similarity = model_similarity[model_similarity["layer"] == layer]
        matrix = layer_similarity.pivot(
            index="direction_a", columns="direction_b", values="absolute_cosine"
        )
        figure, axis = plt.subplots(figsize=(8, 6))
        sns.heatmap(matrix, vmin=0, vmax=1, cmap="viridis", annot=True, fmt=".2f", ax=axis)
        axis.set_title(f"{model_name}: absolute cosine at boundary {layer}")
        axis.set_xlabel("Direction")
        axis.set_ylabel("Direction")
        figure.tight_layout()
        figure_path = model_figure_dir / f"absolute_cosine_boundary_{int(layer):02d}.png"
        figure.savefig(figure_path, dpi=180, bbox_inches="tight")
        plt.show()
        plt.close(figure)
        print("Saved:", figure_path)

Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/gpt2-small/absolute_cosine_boundary_01.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/gpt2-small/absolute_cosine_boundary_06.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/gpt2-small/absolute_cosine_boundary_12.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/qwen-0.6b/absolute_cosine_boundary_01.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/qwen-0.6b/absolute_cosine_boundary_14.png
Saved: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures/qwen-0.6b/absolute_cosine_boundary_28.png


## 14. Final Drive locations

In [14]:
print("Completed run:", RUN_LAYOUT.run_id)
print("Manifest:     ", RUN_LAYOUT.manifest_path)
print("Configuration:", RUN_LAYOUT.root / "requested_config.yaml")
print("Environment:  ", RUN_LAYOUT.root / "environment.json")
print("Directions:   ", RUN_LAYOUT.directions_dir)
print("Results:      ", RUN_LAYOUT.results_dir)
print("Figures:      ", RUN_LAYOUT.figures_dir)

Completed run: 2026-09-05_23-18_CDT
Manifest:      /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/run_manifest.json
Configuration: /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/requested_config.yaml
Environment:   /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/environment.json
Directions:    /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/directions
Results:       /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/results
Figures:       /content/drive/MyDrive/sentiment-geometry/sentiment-position-comparison/runs/2026-09-05_23-18_CDT/figures


In [ ]:
_RUNTIME_SECRETS.clear()